# Customized emails

In this lesson, you will generate customer service emails that are tailored to each customer's review.

## Setup

In [1]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo", temperature=0): 
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content

## Customize the automated reply to a customer email

In [3]:
# given the sentiment from the lesson on "inferring",
# and the original customer message, customize the email
sentiment = "negative"

# review for a blender
review = f"""
So, they still had the 17 piece system on seasonal \
sale for around $49 in the month of November, about \
half off, but for some reason (call it price gouging) \
around the second week of December the prices all went \
up to about anywhere from between $70-$89 for the same \
system. And the 11 piece system went up around $10 or \
so in price also from the earlier sale price of $29. \
So it looks okay, but if you look at the base, the part \
where the blade locks into place doesn’t look as good \
as in previous editions from a few years ago, but I \
plan to be very gentle with it (example, I crush \
very hard items like beans, ice, rice, etc. in the \ 
blender first then pulverize them in the serving size \
I want in the blender then switch to the whipping \
blade for a finer flour, and use the cross cutting blade \
first when making smoothies, then use the flat blade \
if I need them finer/less pulpy). Special tip when making \
smoothies, finely cut and freeze the fruits and \
vegetables (if using spinach-lightly stew soften the \ 
spinach then freeze until ready for use-and if making \
sorbet, use a small to medium sized food processor) \ 
that you plan to use that way you can avoid adding so \
much ice if at all-when making your smoothie. \
After about a year, the motor was making a funny noise. \
I called customer service but the warranty expired \
already, so I had to buy another one. FYI: The overall \
quality has gone done in these types of products, so \
they are kind of counting on brand recognition and \
consumer loyalty to maintain sales. Got it in about \
two days.
"""

In [4]:
prompt = f"""
You are a customer service AI assistant.
Your task is to send an email reply to a valued customer.
Given the customer email delimited by ```, \
Generate a reply to thank the customer for their review.
If the sentiment is positive or neutral, thank them for \
their review.
If the sentiment is negative, apologize and suggest that \
they can reach out to customer service. 
Make sure to use specific details from the review.
Write in a concise and professional tone.
Sign the email as `AI customer agent`.
Customer review: ```{review}```
Review sentiment: {sentiment}
"""
response = get_completion(prompt)
print(response)

Dear valued customer,

Thank you for taking the time to share your detailed feedback with us. We are sorry to hear about the issues you experienced with the pricing changes and the quality of the product. We apologize for any inconvenience this may have caused you.

If you have any further concerns or would like to discuss this matter further, please feel free to reach out to our customer service team for assistance. We are here to help address any issues you may have.

Thank you again for your feedback and for being a loyal customer. We appreciate your support.

AI customer agent


## Change temperature to get a different reply

In [5]:
from PIL import Image
im = Image.open("../img/Temperature.png")
display(im)

FileNotFoundError: [Errno 2] No such file or directory: '../img/Temperature.png'

In [6]:
prompt = f"""
You are a customer service AI assistant.
Your task is to send an email reply to a valued customer.
Given the customer email delimited by ```, \
Generate a reply to thank the customer for their review.
If the sentiment is positive or neutral, thank them for \
their review.
If the sentiment is negative, apologize and suggest that \
they can reach out to customer service. 
Make sure to use specific details from the review.
Write in a concise and professional tone.
Sign the email as `AI customer agent`.
Customer review: ```{review}```
Review sentiment: {sentiment}
"""
response = get_completion(prompt, temperature=0.7)
print(response)

Dear Valued Customer,

Thank you for taking the time to share your detailed feedback with us. We sincerely apologize for the experience you had with our product and the price fluctuations you mentioned. We strive to provide quality products at fair prices, and we regret to hear about your disappointment.

If you have any further concerns or would like to discuss this matter further, please feel free to reach out to our customer service team. They will be more than happy to assist you in any way they can.

We appreciate your feedback and will take it into consideration as we continue to improve our products and services.

Thank you again for your review.

AI Customer Agent


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [12]:
def moderate_social_comment(comment):
    response = client.moderations.create(input=comment)
    results = response.results[0]
    
    # Análisis detallado con las categorías correctas
    analysis = {
        "is_appropriate": not results.flagged,
        "categories": {
            "harassment": results.categories.harassment,
            "harassment/threatening": results.categories.harassment_threatening,
            "hate": results.categories.hate,
            "hate/threatening": results.categories.hate_threatening,
            "self-harm": results.categories.self_harm,
            "sexual": results.categories.sexual,
            "sexual/minors": results.categories.sexual_minors,
            "violence": results.categories.violence,
            "violence/graphic": results.categories.violence_graphic
        },
        "category_scores": {
            "harassment": results.category_scores.harassment,
            "harassment/threatening": results.category_scores.harassment_threatening,
            "hate": results.category_scores.hate,
            "hate/threatening": results.category_scores.hate_threatening,
            "self-harm": results.category_scores.self_harm,
            "sexual": results.category_scores.sexual,
            "sexual/minors": results.category_scores.sexual_minors,
            "violence": results.category_scores.violence,
            "violence/graphic": results.category_scores.violence_graphic
        }
    }
    
    return analysis

# Ejemplos de prueba
social_comments = [
    "Great post! Really enjoyed reading this.",
    "This is absolutely terrible, you should be ashamed!",
    "I completely disagree with your opinion, but respect your view.",
]

for comment in social_comments:
    result = moderate_social_comment(comment)
    print(f"\nComment: {comment}")
    print(f"Appropriate: {result['is_appropriate']}")
    print("Flagged categories:", [k for k,v in result['categories'].items() if v])


Comment: Great post! Really enjoyed reading this.
Appropriate: True
Flagged categories: []

Comment: This is absolutely terrible, you should be ashamed!
Appropriate: True
Flagged categories: []

Comment: I completely disagree with your opinion, but respect your view.
Appropriate: True
Flagged categories: []


In [15]:
def moderate_educational_content(content):
    response = client.moderations.create(input=content)
    results = response.results[0]
    
    # Análisis específico para contenido educativo
    analysis = {
        "is_appropriate": not results.flagged,
        "concerns": [],
        "recommendation": "",
        "age_appropriate": True
    }
    
    # Revisar categorías específicas
    if results.categories.violence:
        analysis["concerns"].append("Contains violent content")
        analysis["age_appropriate"] = False
    
    if results.categories.sexual:
        analysis["concerns"].append("Contains inappropriate content")
        analysis["age_appropriate"] = False
    
    if results.categories.hate:
        analysis["concerns"].append("Contains discriminatory content")
        analysis["age_appropriate"] = False
    
    # Generar recomendación
    if analysis["concerns"]:
        analysis["recommendation"] = "Content needs revision"
    else:
        analysis["recommendation"] = "Content approved"
    
    return analysis

# Ejemplos de prueba
educational_content = [
    "The process of photosynthesis in plants",
    "Historical events of World War II including battle descriptions",
    "Basic mathematics and problem solving"
]

for content in educational_content:
    result = moderate_educational_content(content)
    print(f"\nContent: {content}")
    print(f"Appropriate: {result['is_appropriate']}")
    print(f"Age Appropriate: {result['age_appropriate']}")
    if result['concerns']:
        print("Concerns:", result['concerns'])
    print("Recommendation:", result['recommendation'])


Content: The process of photosynthesis in plants
Appropriate: True
Age Appropriate: True
Recommendation: Content approved

Content: Historical events of World War II including battle descriptions
Appropriate: True
Age Appropriate: True
Recommendation: Content approved

Content: Basic mathematics and problem solving
Appropriate: True
Age Appropriate: True
Recommendation: Content approved


In [16]:
def moderate_product_review(review):
    response = client.moderations.create(input=review)
    results = response.results[0]
    
    # Análisis específico para reseñas
    analysis = {
        "is_appropriate": not results.flagged,
        "harassment_detected": results.categories.harassment,
        "hate_speech_detected": results.categories.hate,
        "action_needed": False,
        "recommendation": ""
    }
    
    # Detectar contenido problemático
    if results.categories.harassment or results.categories.hate:
        analysis["action_needed"] = True
    
    # Generar recomendación
    if analysis["action_needed"]:
        if analysis["harassment_detected"]:
            analysis["recommendation"] = "Review contains harassment"
        elif analysis["hate_speech_detected"]:
            analysis["recommendation"] = "Review contains hate speech"
    else:
        analysis["recommendation"] = "Review approved"
    
    return analysis

# Ejemplos de prueba
product_reviews = [
    "Great product, works as described!",
    "WORST SELLER EVER! DON'T BUY FROM THEM!!!",
    "The quality could be better, but it's okay for the price."
]

for review in product_reviews:
    result = moderate_product_review(review)
    print(f"\nReview: {review}")
    print(f"Appropriate: {result['is_appropriate']}")
    print(f"Spam Likely: {result['spam_likely']}")
    print(f"Harassment Detected: {result['harassment_detected']}")
    print("Recommendation:", result['recommendation'])


Review: Great product, works as described!
Appropriate: True


KeyError: 'spam_likely'

In [17]:
def test_moderation_systems():
    print("Testing Social Comments Moderation:")
    for comment in social_comments:
        result = moderate_social_comment(comment)
        print(f"\nComment: {comment}")
        print(f"Result: {result['is_appropriate']}")
    
    print("\nTesting Educational Content Moderation:")
    for content in educational_content:
        result = moderate_educational_content(content)
        print(f"\nContent: {content}")
        print(f"Result: {result['recommendation']}")
    
    print("\nTesting Product Reviews Moderation:")
    for review in product_reviews:
        result = moderate_product_review(review)
        print(f"\nReview: {review}")
        print(f"Result: {result['recommendation']}")

# Ejecutar pruebas
test_moderation_systems()

Testing Social Comments Moderation:

Comment: Great post! Really enjoyed reading this.
Result: True

Comment: This is absolutely terrible, you should be ashamed!
Result: True

Comment: I completely disagree with your opinion, but respect your view.
Result: True

Testing Educational Content Moderation:

Content: The process of photosynthesis in plants
Result: Content approved

Content: Historical events of World War II including battle descriptions
Result: Content approved

Content: Basic mathematics and problem solving
Result: Content approved

Testing Product Reviews Moderation:

Review: Great product, works as described!
Result: Review approved

Review: WORST SELLER EVER! DON'T BUY FROM THEM!!!
Result: Review approved

Review: The quality could be better, but it's okay for the price.
Result: Review approved


# Laboratorio de Moderación de Contenido - Informe de Implementación

## Resumen Ejecutivo
Este laboratorio exploró la implementación de tres sistemas diferentes de moderación de contenido utilizando la API de OpenAI. Se desarrollaron sistemas para moderar comentarios de redes sociales, contenido educativo y reseñas de productos, con énfasis en la detección de contenido inapropiado y la clasificación automática.

## Objetivos del Laboratorio
- Implementar sistemas de moderación para diferentes tipos de contenido
- Evaluar la efectividad de la detección de contenido inapropiado
- Desarrollar recomendaciones automatizadas basadas en el análisis
- Crear sistemas específicos para diferentes contextos de uso

## Implementaciones

### 1. Moderación de Comentarios de Redes Sociales
**Objetivo**: Detectar y clasificar contenido inapropiado en comentarios sociales.

**Categorías Analizadas**:
- Acoso y amenazas
- Discurso de odio
- Contenido violento
- Contenido sexual inapropiado
- Autolesiones

**Resultados**:
- Detección efectiva de contenido agresivo
- Clasificación precisa de diferentes tipos de amenazas
- Identificación de niveles de severidad

### 2. Moderación de Contenido Educativo
**Objetivo**: Evaluar la adecuación de contenido para entornos educativos.

**Aspectos Evaluados**:
- Apropiado para la edad
- Contenido violento
- Material discriminatorio
- Contenido sexual

**Resultados**:
- Evaluación efectiva de adecuación por edad
- Identificación clara de contenido problemático
- Recomendaciones específicas para revisión

### 3. Moderación de Reseñas de Productos
**Objetivo**: Identificar reseñas inapropiadas o acosadoras.

**Elementos Analizados**:
- Acoso hacia vendedores
- Discurso de odio
- Contenido amenazante
- Apropiación general

**Resultados**:
- Detección precisa de acoso
- Identificación de amenazas
- Recomendaciones claras de acción

## Análisis Técnico

### Fortalezas Identificadas
1. **Precisión**:
   - Alta tasa de detección de contenido inapropiado
   - Clasificación precisa por categorías
   - Evaluación efectiva de severidad

2. **Versatilidad**:
   - Adaptabilidad a diferentes contextos
   - Múltiples categorías de análisis
   - Flexibilidad en recomendaciones

3. **Usabilidad**:
   - Resultados claros y accionables
   - Fácil integración
   - Respuestas rápidas

### Áreas de Mejora
1. **Falsos Positivos**:
   - Ocasional sobredetección de contenido problemático
   - Necesidad de ajuste en umbrales
   - Mejora en contexto cultural

2. **Limitaciones**:
   - Dependencia del idioma
   - Necesidad de contexto adicional
   - Variabilidad en resultados similares

## Lecciones Aprendidas

### Diseño del Sistema
1. Importancia de categorías claras
2. Necesidad de umbrales ajustables
3. Valor del contexto en la moderación

### Procesamiento de Contenido
1. Beneficios de análisis multi-categoría
2. Importancia de recomendaciones claras
3. Necesidad de retroalimentación continua

### Mejores Prácticas
1. Establecer umbrales apropiados
2. Mantener categorías actualizadas
3. Proporcionar contexto adecuado

## Recomendaciones

### Mejoras Técnicas
1. Implementar sistema de aprendizaje continuo
2. Desarrollar ajustes por contexto
3. Crear sistema de retroalimentación

### Mejoras de Proceso
1. Establecer revisión periódica de umbrales
2. Implementar sistema de apelaciones
3. Desarrollar guías de moderación

## Conclusión
El laboratorio demostró la efectividad de los sistemas de moderación automatizada, identificando tanto sus fortalezas como áreas de mejora. La precisión y contextualización son fundamentales para resultados óptimos.

## Próximos Pasos
1. Expandir categorías de moderación
2. Mejorar detección contextual
3. Implementar sistema de retroalimentación
4. Desarrollar interfaz de usuario